# AXI test 6: does it hold at one million tokens? (hosted model, 3,600 steps)
Model: **Gemini 3.5 Flash Lite** (Google's small, fast tier; a current model, not a Pro model). One long job: **3,600 requests in a row** about a small service graph, about 1.2 to 1.4 million tokens. At every step the model is shown only the exact current state and one request; the framework (the scope filter, `scope_m7`) decides what to keep, and the held state is compared with the right one at every step. The same stream, the same 13 reading rules and the same thresholds as the Qwen run (`axi_million_test.ipynb`). Predictions were written and locked before this run in `docs/MILLION_TOKEN_TEST_GEMINI.md`.

* **Run 1:** 3,600 calls. **Contrast:** 900 more (the first 300 requests again with blind, gate and repair_m6). **Run 2** (optional): 3,600 more.
* **No GPU.** Use the free CPU runtime (Runtime menu -> *Change runtime type* -> CPU).
* **Your key** lives in Colab Secrets (Cell 1). It is never printed or saved in a file. Never paste it in the chat.

**Free or paid: your choice, and the notebook never turns billing on.**
* *Free key:* about 480 calls a day (the notebook stops by itself and says so). Run 1 takes **about 8 days**, one cell a day; the contrast about 2 more. The limit resets at midnight Pacific time: **12:30 pm in India until 1 November, 1:30 pm after.** The words test and the confirmation test use the same daily quota.
* *Paid key (billing turned on by you in Google AI Studio):* set `PAID = True` in Cell 5. The whole of Run 1 takes **about an hour**. Thinking tokens are billed like output tokens. The cell prints the real token counts so you can check them against your bill, and a **hard cap of 2,000,000 tokens on each run** (Run 1, each contrast file, Run 2) stops that run by itself. Prices are not built in: type the two prices from the AI Studio price page into Cell 5 if you want a dollar figure.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. If it stops for any reason, open this notebook again, run **Cells 2 to 5**, then run the cell you were on again: it continues from the last saved step.

### Cell 1: put your key into Colab Secrets (once; no code to run)
1. In Google AI Studio (`aistudio.google.com`), open **API Keys** in the left menu and copy your key.
2. Here in Colab, tap the **key icon** in the left bar (*Secrets*) -> **Add new secret**.
3. Name: `GEMINI_API_KEY` (exactly). Value: paste the key. Switch **Notebook access** on.
Do not paste the key into a cell or into the chat.

### Cell 2: upload the code. A button 'Choose files' appears: pick **only** `axi-colab-million.zip`

In [ ]:
from google.colab import files
up = files.upload()
cands = [n for n in up if n.endswith(".zip") and "million" in n]
assert len(cands) == 1, f"I need exactly one file named like axi-colab-million.zip. I got: {list(up)}"
name = cands[0]
print("using", name)
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and every reply are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_million_gemini")

### Cell 5: settings, and connect to the model. Read the four lines at the top; change them only if you mean to

In [ ]:
import json, time
from collections import deque
from google.colab import userdata
from axi.experiments.api_llm import QuotaStop, ServiceBusy, build_gemini_llm
from axi.experiments.million import (run_million, verdict_million, print_million, print_verdict_million, smoke_report,
                                     SEEDS, STEPS, CONTRAST_STEPS, CONTRAST_POLICIES, MAIN_POLICY, GEMINI_MODEL, GEMINI_TOKEN_CAP, GEMINI_FREE_DAILY)

PAID = False       # True ONLY after you turned billing on for this key in Google AI Studio. Then there is no daily stop and calls are not spaced out.
THINKING = None    # None = the model's own default (as in the earlier Flash Lite tests). Cell 6 tells you if it should be "minimal" instead.
PRICE_IN = None    # dollars per 1,000,000 INPUT tokens, from the AI Studio price page. Leave None to see tokens only.
PRICE_OUT = None   # dollars per 1,000,000 OUTPUT tokens (thinking tokens are billed as output). Leave None to see tokens only.

try: KEY = userdata.get("GEMINI_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: add a Secret named GEMINI_API_KEY and switch Notebook access on.")
TAG = str(THINKING or "default")
CACHE = "axi_million_cache_%s.jsonl" % TAG
print("reply file:", store.restore(CACHE))
llm, MODEL = build_gemini_llm(KEY, GEMINI_MODEL, store.path(CACHE), daily_limit=None if PAID else GEMINI_FREE_DAILY, min_interval=0.3 if PAID else 4.5,
                              on_write=lambda: store.persist(CACHE), thinking=THINKING)
META = dict(model=MODEL, thinking=THINKING)
print("paid:", PAID, "| thinking:", THINKING or "model default", "| calls today:", llm.calls_today(), "| token cap:", f"{GEMINI_TOKEN_CAP:,}")
RESET = "The free quota resets at midnight Pacific time: 12:30 pm in India until 1 November, 1:30 pm after. Then open this notebook again, run Cells 2 to 5, and run this cell again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once. Do NOT run it again and again: Google counts failed tries against the free daily limit."
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e)); print(BUSY if isinstance(e, ServiceBusy) else RESET); return None

def make_progress(name):
    T0, first, recent = time.time(), [None], deque(maxlen=100)
    def cb(t, n, rec):
        if first[0] is None: first[0] = t
        recent.append(rec)
        if t % 100 == 0 or t == n:
            k = len(recent); el = time.time() - T0
            print(f"step {t}/{n} | {el/60:.0f} min so far | calls today {llm.calls_today()} | last {k}: exact {100*sum(r['exact'] for r in recent)/k:.0f}%, mean distance "
                  f"{sum(r['distance'] for r in recent)/k:.2f}, {sum(r['seconds'] for r in recent)/k:.2f} s a call, {sum(r['think'] for r in recent)/k:.0f} thinking tokens a step")
    return cb

def save(res, name):
    path = store.path(name + ".json"); json.dump(res, open(path, "w")); store.persist(name + ".json"); return path

def run_file(name, seed, steps, policy):
    print("progress file:", store.restore(name + ".jsonl"))
    def persist_cb(inner):
        def cb(t, n, rec):
            inner(t, n, rec)
            if t % 25 == 0 or t == n: store.persist(name + ".jsonl")
        return cb
    return go(lambda: run_million(llm, seed, steps, policy, store.path(name + ".jsonl"), meta=META, progress=persist_cb(make_progress(name)), max_tokens=GEMINI_TOKEN_CAP))

def stop_note(res):
    if res and res["stopped"] == "token_cap": print("STOPPED by the token cap of %s tokens. This is the guard on the bill; tell me before you raise it." % f"{GEMINI_TOKEN_CAP:,}"); return None
    return res

### Cell 6: quick check, 20 steps (about 1 to 2 minutes; 20 of today's calls). It prints the tokens a step, the thinking tokens, and what the whole run would take. If it says NOTE, read it before going on

In [ ]:
tag = "axi_million_check_%s" % TAG
smoke = run_file(tag, 99, 20, MAIN_POLICY)
if smoke:
    print_million(smoke["summary"], "check, 20 steps")
    rep = smoke_report(smoke["summary"], STEPS, PRICE_IN, PRICE_OUT)
    if rep["steps"] == 0: print("NOTE:", rep["advice"][0])
    else:
        print(f"\n{rep['tokens_per_step']:.0f} tokens a step ({rep['think_per_step']:.1f} of them thinking), {rep['seconds_per_step']:.2f} s a call: {STEPS} steps would be about {rep['projected_tokens'] / 1e6:.2f} million tokens, "
              f"{rep['projected_minutes']:.0f} minutes of calls" + (f", about ${rep['projected_cost']:.2f} at the prices you typed" if rep["projected_cost"] is not None else ""))
        if not PAID: print(f"On the free key that is about {-(-STEPS // GEMINI_FREE_DAILY)} days for Run 1, then about {-(-3 * CONTRAST_STEPS // GEMINI_FREE_DAILY)} for the contrast.")
        for a in rep["advice"]: print("NOTE:", a)
    for r in smoke["rows"][:6]: print(r["t"], r["kind"], "|", repr(r["text"])[:90])

### Cell 7: RUN 1. 3,600 steps, scope_m7. A line is printed every 100 steps; progress and every reply are saved to Drive. On the free key it stops for the day at 480 calls: run it again the next day

In [ ]:
name = "axi_million_gemini_run1_seed%d_%s" % (SEEDS[0], TAG)
res1 = stop_note(run_file(name, SEEDS[0], STEPS, MAIN_POLICY))
if res1:
    print_million(res1["summary"], "RUN 1, scope_m7, seed %d, %s" % (SEEDS[0], MODEL))
    print_verdict_million(verdict_million(res1), "RUN 1 (lines 4, 12 and 13 need the next cells)")
    print("totals from the service:", llm.usage())
    files.download(save(res1, name)); files.download(store.path(CACHE))

### Cell 8: CONTRAST. The first 300 requests again with blind, gate and repair_m6 (900 calls; on the free key about 2 days). Also prints the reading-rule lines

In [ ]:
assert res1, "Run 1 has not finished yet. Run Cell 7 again first."
contrast = {}
for p in CONTRAST_POLICIES:
    cname = "axi_million_gemini_contrast_%s_seed%d_%s" % (p, SEEDS[0], TAG)
    r = stop_note(run_file(cname, SEEDS[0], CONTRAST_STEPS, p))
    if not r: break
    contrast[p] = r; print_million(r["summary"], "CONTRAST, %s" % p)
if len(contrast) == len(CONTRAST_POLICIES):
    print_verdict_million(verdict_million(res1, contrast), "RUN 1 and the contrast")
    files.download(save(contrast, "axi_million_gemini_contrast_seed%d_%s" % (SEEDS[0], TAG)))

### Cell 9 (optional): RUN 2, the second seed, for 'two million' (3,600 more calls). Only if you have the time or the paid key

In [ ]:
assert res1 and len(contrast) == len(CONTRAST_POLICIES), "Finish Cells 7 and 8 first."
name2 = "axi_million_gemini_run2_seed%d_%s" % (SEEDS[1], TAG)
res2 = stop_note(run_file(name2, SEEDS[1], STEPS, MAIN_POLICY))
if res2:
    print_million(res2["summary"], "RUN 2, scope_m7, seed %d, %s" % (SEEDS[1], MODEL))
    print_verdict_million(verdict_million(res1, contrast, res2), "RUN 1, the contrast and RUN 2")
    print("totals from the service:", llm.usage())
    files.download(save(res2, name2)); files.download(store.path(CACHE))

### Done
Files downloaded: `axi_million_gemini_run1_seed20261021_default.json` (the end of the name says `minimal` if you changed THINKING), `axi_million_gemini_contrast_seed20261021_default.json`, `axi_million_cache_default.jsonl` (and the run-2 file if you ran Cell 9). If a download did not start, the same files are in your Google Drive, in the folder `axi_million_gemini`. Upload them to the chat, with a screenshot of the printed tables and the reading-rule lines. Do not paste any keys anywhere.